# 00 – Kiểm tra môi trường Deep Learning (ViSoBERT & PhoBERT)

**Chuẩn bị trước khi chạy**
- **Kaggle:** *Settings → Accelerator → GPU T4 x2* (hoặc T4), bật *Settings → Internet → On* để tải mô hình từ Hugging Face.
- **Colab:** *Runtime → Change runtime type → T4 GPU*.
- Sau đó *Run All*. Ô 1 phải in ra `CUDA: True`.

In [1]:
# 1. Kiểm tra GPU
!nvidia-smi
import torch, platform
print("Python :", platform.python_version())
print("Torch  :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
else:
    print("⚠️ Chưa bật GPU – xem hướng dẫn ở ô đầu notebook.")

Tue Nov 24 15:47:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Cài thư viện
Kaggle / Colab đã có sẵn `torch` **đúng bản CUDA** → **không** `pip install torch` lại (dễ làm hỏng CUDA).
- `sentencepiece`, `protobuf`: tokenizer của ViSoBERT cần.
- `pyvi`, `underthesea`: tách từ cho PhoBERT (bản chính thức dùng VnCoreNLP – cần Java, sẽ thử ở tuần 2 khi viết `tokenizer.py`).

In [2]:
!pip -q install transformers==5.24.0 sentencepiece protobuf pyvi underthesea
import transformers
print("transformers:", transformers.__version__)

transformers: 5.24.0


## 3. Cấu hình
9 nhãn BIO theo `PROJECT_PLAN.md` §7.1.

In [3]:
import time, gc
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LABELS = ["O", "B-ASP", "I-ASP", "B-POS", "I-POS", "B-NEU", "I-NEU", "B-NEG", "I-NEG"]
NUM_LABELS = len(LABELS)            # = 9
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}

MODELS = {
    "visobert": "uitnlp/visobert",
    "phobert":  "vinai/phobert-base-v2",
}

# Câu mẫu: có câu chuẩn, câu teencode, câu không dấu, emoji
SAMPLES = [
    "Áo đẹp nhưng chất vải thì mỏng quá, shop tư vấn nhiệt tình. Giao hơi lâu.",
    "sp dùng ok, giao hàng nhanh, đóng gói cẩn thận 👍",
    "pin trâu nhưng màn hình hơi tối, ko đáng tiền lắm",
    "hàng dc lắm nha mn, sẽ ủng hộ shop tiếp",
    "son len mau dep, mui thom nhung hop hoi mop",
    "Giao nhanh thật, 2 tuần mới tới 🙂",
]
print(DEVICE, NUM_LABELS)

cuda 9


## 4. Hàm tải mô hình và chạy forward

In [4]:
def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForTokenClassification.from_pretrained(
        name, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id
    ).to(DEVICE).eval()
    return tok, model

def show_tokens(tok, text):
    enc = tok(text)
    toks = tok.convert_ids_to_tokens(enc["input_ids"])
    n_unk = sum(i == tok.unk_token_id for i in enc["input_ids"])
    print(f"  [{len(toks):>3} token | unk={n_unk}] {toks}")
    return len(toks), n_unk

@torch.no_grad()
def forward_one(tok, model, text):
    enc = tok(text, return_tensors="pt").to(DEVICE)
    logits = model(**enc).logits
    assert logits.shape == (1, enc["input_ids"].shape[1], NUM_LABELS), logits.shape
    return tuple(logits.shape)

## 5. ViSoBERT
Cảnh báo *"Some weights ... newly initialized: classifier..."* khi tải là **bình thường** (đầu 9 nhãn chưa train).

Dùng **văn bản gốc (lớp 1)**, không tách từ, không chuẩn hoá teencode (§6). Quan sát: `ko`, `dc`, `sp`, emoji, câu không dấu bị tách subword thế nào.

In [5]:
results = []

name = MODELS["visobert"]
tok, model = load(name)
print(name, "| fast tokenizer:", tok.is_fast, "| vocab:", tok.vocab_size,
      "| hidden:", model.config.hidden_size, "| max_pos:", model.config.max_position_embeddings)
for s in SAMPLES:
    print("•", s)
    show_tokens(tok, s)
    print("  logits:", forward_one(tok, model, s))

# Các từ teencode / emoji riêng lẻ
for w in ["ko", "k", "dc", "đc", "sp", "mn", "đẹppppp", "👍", "😡", "chất vải"]:
    print(f"{w!r:>12} ->", tok.tokenize(w))

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: uitnlp/visobert
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


uitnlp/visobert | fast tokenizer: True | vocab: 15002 | hidden: 768 | max_pos: 514
• Áo đẹp nhưng chất vải thì mỏng quá, shop tư vấn nhiệt tình. Giao hơi lâu.
  [ 32 token | unk=0] ['<s>', '▁Áo', '▁đ', 'ẹp', '▁như', 'ng', '▁ch', 'ất', '▁v', 'ải', '▁thì', '▁m', 'ỏng', '▁quá', ',', '▁shop', '▁tư', '▁v', 'ấn', '▁nh', 'iệt', '▁t', 'ình', '.', '▁Gi', 'ao', '▁h', 'ơi', '▁l', 'âu', '.', '</s>']


  logits: (1, 32, 9)
• sp dùng ok, giao hàng nhanh, đóng gói cẩn thận 👍
  [ 23 token | unk=0] ['<s>', '▁sp', '▁d', 'ùng', '▁ok', ',', '▁gi', 'ao', '▁h', 'àng', '▁nha', 'nh', ',', '▁đó', 'ng', '▁g', 'ói', '▁c', 'ẩn', '▁th', 'ận', '▁👍', '</s>']
  logits: (1, 23, 9)
• pin trâu nhưng màn hình hơi tối, ko đáng tiền lắm
  [ 24 token | unk=0] ['<s>', '▁p', 'in', '▁tr', 'âu', '▁như', 'ng', '▁m', 'àn', '▁h', 'ình', '▁h', 'ơi', '▁t', 'ối', ',', '▁ko', '▁đ', 'áng', '▁t', 'iền', '▁l', 'ắm', '</s>']
  logits: (1, 24, 9)
• hàng dc lắm nha mn, sẽ ủng hộ shop tiếp
  [ 15 token | unk=0] ['<s>', '▁h', 'àng', '▁dc', '▁l', 'ắm', '▁nha', '▁mn', ',', '▁sẽ', '▁ủng', '▁hộ', '▁shop', '▁tiếp', '</s>']
  logits: (1, 15, 9)
• son len mau dep, mui thom nhung hop hoi mop
  [ 23 token | unk=0] ['<s>', '▁s', 'on', '▁l', 'en', '▁m', 'au', '▁d', 'ep', ',', '▁m', 'ui', '▁th', 'om', '▁nh', 'ung', '▁h', 'op', '▁h', 'oi', '▁m', 'op', '</s>']
  logits: (1, 23, 9)
• Giao nhanh thật, 2 tuần mới tới 🙂
  [ 17 token | unk=0] ['<

**Ghi nhận ViSoBERT:**

- **Có giữ dấu tiếng Việt không?** Có. Cả 6 câu đều không có từ nào bị lỗi (`<unk>` = 0). Nhưng mô hình hay cắt 1 chữ thành 2 mảnh, ví dụ `đẹp` → `đ` + `ẹp`.
- **Teencode và emoji?** `ko`, `dc`, `sp`, `mn` giữ nguyên là 1 token → mô hình đã hiểu teencode, không cần đổi `ko` thành `không`. Emoji 👍 😡 cũng là 1 token riêng.
- **Câu không dấu?** Bị cắt nhỏ tương đương câu có dấu (khoảng 2 token/từ), không bị lỗi.
- **`tok.is_fast` = True** → tuần 2 dùng được `word_ids()` để gán nhãn cho từng từ.

Kết quả: chạy không lỗi, đầu ra đúng dạng `[1, số_token, 9]`.

## 6. PhoBERT
PhoBERT được huấn luyện trên văn bản **đã tách từ** (`chất_vải`). So sánh 3 cách đưa vào: câu gốc / tách bằng `pyvi` / tách bằng `underthesea`.
Chỉ số cần nhìn: số token và số `<unk>`.

In [6]:
# Giải phóng ViSoBERT trước khi tải PhoBERT
del model; gc.collect(); torch.cuda.empty_cache()

from pyvi import ViTokenizer
from underthesea import word_tokenize

def seg_pyvi(s): return ViTokenizer.tokenize(s)
def seg_uts(s):  return word_tokenize(s, format="text")

name = MODELS["phobert"]
tok, model = load(name)
print(name, "| fast tokenizer:", tok.is_fast, "| vocab:", tok.vocab_size,
      "| hidden:", model.config.hidden_size, "| max_pos:", model.config.max_position_embeddings)

rows = []
for s in SAMPLES:
    print("•", s)
    for how, txt in [("gốc", s), ("pyvi", seg_pyvi(s)), ("underthesea", seg_uts(s))]:
        print(f"  {how:<11}: {txt}")
        n, unk = show_tokens(tok, txt)
        rows.append({"câu": s[:30] + "…", "cách": how, "số token": n, "unk": unk})
    print("  logits:", forward_one(tok, model, seg_pyvi(s)))

pd.DataFrame(rows).pivot_table(index="câu", columns="cách", values=["số token", "unk"])

[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


vinai/phobert-base-v2 | fast tokenizer: False | vocab: 64000 | hidden: 768 | max_pos: 258
• Áo đẹp nhưng chất vải thì mỏng quá, shop tư vấn nhiệt tình. Giao hơi lâu.
  gốc        : Áo đẹp nhưng chất vải thì mỏng quá, shop tư vấn nhiệt tình. Giao hơi lâu.
  [ 21 token | unk=0] ['<s>', 'Áo', 'đẹp', 'nhưng', 'chất', 'vải', 'thì', 'mỏng', 'quá@@', ',', 'shop', 'tư', 'vấn', 'nhiệt', 'tì@@', 'nh.', 'Giao', 'hơi', 'lâ@@', 'u.', '</s>']
  pyvi       : Áo đẹp nhưng chất vải thì mỏng quá , shop tư_vấn nhiệt_tình . Giao hơi lâu .
  [ 19 token | unk=0] ['<s>', 'Áo', 'đẹp', 'nhưng', 'chất', 'vải', 'thì', 'mỏng', 'quá', ',', 'shop', 'tư_vấn', 'nhiệt_tình', '.', 'Giao', 'hơi', 'lâu', '.', '</s>']
  underthesea: Áo đẹp nhưng chất vải thì mỏng quá , shop tư_vấn nhiệt_tình . Giao hơi lâu .
  [ 19 token | unk=0] ['<s>', 'Áo', 'đẹp', 'nhưng', 'chất', 'vải', 'thì', 'mỏng', 'quá', ',', 'shop', 'tư_vấn', 'nhiệt_tình', '.', 'Giao', 'hơi', 'lâu', '.', '</s>']
  logits: (1, 19, 9)
• sp dùng ok, giao hàng nhanh,

số token                    unk       \
cách                                 gốc  pyvi underthesea  gốc pyvi   
câu                                                                    
Giao nhanh thật, 2 tuần mới tớ…     12.0  11.0        11.0  1.0  1.0   
hàng dc lắm nha mn, sẽ ủng hộ …     15.0  14.0        14.0  0.0  0.0   
pin trâu nhưng màn hình hơi tố…     15.0  13.0        13.0  0.0  0.0   
son len mau dep, mui thom nhun…     15.0  15.0        16.0  0.0  0.0   
sp dùng ok, giao hàng nhanh, đ…     15.0  13.0        13.0  1.0  1.0   
Áo đẹp nhưng chất vải thì mỏng…     21.0  19.0        19.0  0.0  0.0   

                                             
cách                            underthesea  
câu                                          
Giao nhanh thật, 2 tuần mới tớ…         1.0  
hàng dc lắm nha mn, sẽ ủng hộ …         0.0  
pin trâu nhưng màn hình hơi tố…         0.0  
son len mau dep, mui thom nhun…         0.0  
sp dùng ok, giao hàng nhanh, đ…         1.0  
Áo đẹp nhưng chất vải thì mỏng…         0.0

**Ghi nhận PhoBERT:**

- **Có cần tách từ không?** Có. Không tách thì `tư vấn`, `nhiệt tình` bị coi là 2 chữ rời, dấu câu dính vào chữ (`nh.`, `u.`). Tách từ xong thì thành `tư_vấn`, `nhiệt_tình`, `màn_hình`, mỗi cụm là 1 token, số token cũng ít hơn (vd câu 1: 21 → 19).
- **pyvi hay underthesea?** Câu bình thường thì 2 cách ra giống nhau. Câu không dấu / teencode thì underthesea ghép sai (`len_mau`, `thom_nhung`, `nha_mn`), pyvi không bị → **dùng pyvi tốt hơn**.
- **Teencode:** `ko`, `sp`, `ok` vẫn là 1 token, nhưng `dc` → `d` + `c`, `mn` → `m` + `n` bị cắt vụn.
- **Emoji:** 👍 🙂 thành `<unk>` (PhoBERT không biết emoji), khác với ViSoBERT.
- **Câu không dấu:** không bị lỗi nhưng hay bị cắt nhỏ (`thom` → `tho` + `m`), vì PhoBERT học từ báo chí có dấu.
- **`tok.is_fast` = False** → không có `word_ids()`, tuần 2 phải tự viết code căn nhãn cho từng từ.
- Câu dài tối đa ~256 token, đủ cho bình luận 10–80 từ.

Kết quả: chạy không lỗi, đầu ra đúng dạng `[1, số_token, 9]`.

## 7. Đo tốc độ & bộ nhớ: batch 32 câu
Đo cho cả 2 mô hình: suy luận (no_grad) và **1 bước train** (forward + backward + optimizer, fp16) – để ước tính thời gian train ở tuần 6.

In [7]:
def benchmark(key, batch_size=32, max_len=128, n_rep=5):
    name = MODELS[key]
    tok, model = load(name)
    texts = (SAMPLES * 10)[:batch_size]
    if key == "phobert":
        texts = [seg_pyvi(t) for t in texts]
    enc = tok(texts, padding="max_length", truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
    sync = torch.cuda.synchronize if DEVICE == "cuda" else (lambda: None)
    if DEVICE == "cuda": torch.cuda.reset_peak_memory_stats()

    # Suy luận
    model.eval()
    with torch.no_grad(), torch.autocast(DEVICE, enabled=(DEVICE == "cuda")):
        model(**enc); sync()                       # warm-up
        t0 = time.time()
        for _ in range(n_rep): out = model(**enc)
        sync(); t_inf = (time.time() - t0) / n_rep
    assert out.logits.shape == (batch_size, max_len, NUM_LABELS)

    # 1 bước train với nhãn giả
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    labels = torch.randint(0, NUM_LABELS, (batch_size, max_len), device=DEVICE)
    def step():
        opt.zero_grad()
        with torch.autocast(DEVICE, enabled=(DEVICE == "cuda")):
            loss = model(**enc, labels=labels).loss
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
    step(); sync()                                 # warm-up
    t0 = time.time()
    for _ in range(n_rep): step()
    sync(); t_train = (time.time() - t0) / n_rep

    peak = torch.cuda.max_memory_allocated() / 1024**3 if DEVICE == "cuda" else float("nan")
    res = {
        "mô hình": name, "fast tokenizer": tok.is_fast, "hidden": model.config.hidden_size,
        "logits batch": tuple(out.logits.shape),
        "suy luận 32 câu (s)": round(t_inf, 3), "1 bước train 32 câu (s)": round(t_train, 3),
        "GPU peak (GB)": round(peak, 2),
        # train 2.000 câu ≈ 63 bước/epoch
        "ước tính 1 epoch 2.000 câu (s)": round(t_train * 2000 / batch_size, 1),
    }
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return res

del model; gc.collect(); torch.cuda.empty_cache()
summary = pd.DataFrame([benchmark("visobert"), benchmark("phobert")])
summary

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: uitnlp/visobert
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_50/1867730962.py:23: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))


[transformers] RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base-v2
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,mô hình,fast tokenizer,hidden,logits batch,suy luận 32 câu (s),1 bước train 32 câu (s),GPU peak (GB),ước tính 1 epoch 2.000 câu (s)
0,uitnlp/visobert,True,768,"(32, 128, 9)",0.055,0.181,2.87,11.3
1,vinai/phobert-base-v2,False,768,"(32, 128, 9)",0.049,0.181,3.29,11.3


## 8. Kết luận

| Hạng mục | ViSoBERT | PhoBERT |
|---|---|---|
| Tải mô hình + forward không lỗi | Có | Có |
| Đầu ra `[1, số_token, 9]` | Có | Có |
| Cần tách từ? | Không | Có (dùng pyvi) |
| Fast tokenizer (`word_ids()`) | Có | Không |
| Hiểu teencode / emoji | Tốt | Kém (emoji bị nhận thành `<unk>`) |
| 1 bước train batch 32 (giây) | 0,18 | 0,18 |
| GPU dùng tối đa (GB) | 2,9 / 15 | 3,3 / 15 |
| Ước tính 1 epoch 2.000 câu | Khoảng 11 giây | Khoảng 11 giây |

- Cả 2 mô hình chạy được trên Kaggle GPU T4, không thiếu thư viện, không tràn bộ nhớ.
- Train nhanh: 10 epoch chỉ mất khoảng 2 phút mỗi mô hình; GPU còn dư nhiều nên có thể tăng batch hoặc độ dài câu.
- ViSoBERT phù hợp với bình luận Tiki hơn (hiểu teencode, emoji, không cần tách từ), đúng với kế hoạch ưu tiên ViSoBERT.

**Ghi chú cho tuần 2 (`tokenizer.py`):** ViSoBERT dùng `word_ids()` để gán nhãn cho token đầu của mỗi từ; PhoBERT phải tách từ bằng pyvi trước và tự viết code căn nhãn (vì không có `word_ids()`).
- Nhãn lưu theo **vị trí ký tự** → với ViSoBERT nên gọi tokenizer với `return_offsets_mapping=True` để biết mỗi token ứng với ký tự nào (chắc hơn `word_ids()`).
- pyvi thêm khoảng trắng trước dấu câu (`quá ,`) và nối từ bằng `_` → với PhoBERT phải căn lại vị trí ký tự của từng từ đã tách về câu gốc.

## Phụ lục – Đếm từ ngắn hay gặp để bổ sung `teencode_dict.json`
Chạy khi đã có `tiki_reviews.csv` (tải từ Drive của nhóm). Trên Kaggle: *Add Input → Upload* file CSV; trên Colab: mount Drive. Sửa `CSV_PATH` cho đúng. Không có file thì ô này tự bỏ qua.

In [8]:
import re, json, collections
from pathlib import Path

CANDIDATES = [
    "data/raw/tiki_reviews.csv", "../data/raw/tiki_reviews.csv",
    "/content/drive/MyDrive/raw/tiki_reviews.csv",
]
CANDIDATES += [str(p) for p in Path("/kaggle/input").rglob("tiki_reviews.csv")] if Path("/kaggle/input").exists() else []
CSV_PATH = next((p for p in CANDIDATES if Path(p).exists()), None)

VN_DIACRITIC = re.compile(r"[àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]")

if CSV_PATH is None:
    print("Chưa có tiki_reviews.csv → bỏ qua phụ lục.")
else:
    df = pd.read_csv(CSV_PATH, usecols=["content"]).dropna()
    print(CSV_PATH, len(df), "review có nội dung")
    cnt = collections.Counter()
    for t in df["content"].str.lower():
        cnt.update(re.findall(r"\w+", t))
    # Ứng viên teencode: từ 1–5 ký tự, không có dấu, không phải số
    cand = [(w, c) for w, c in cnt.most_common(5000)
            if len(w) <= 5 and not VN_DIACRITIC.search(w) and not w.isdigit()]
    print("Top 150 từ ngắn không dấu:")
    print(cand[:150])

    # Độ phủ của teencode_dict hiện tại (nếu notebook chạy trong repo)
    dict_path = next((p for p in ["src/preprocessing/teencode_dict.json", "../src/preprocessing/teencode_dict.json"] if Path(p).exists()), None)
    if dict_path:
        td = json.load(open(dict_path, encoding="utf-8"))
        hit = sum(cnt[k] for k in td)
        print(f"Từ điển: {len(td)} mục, xuất hiện {hit:,} lần trong dữ liệu")
        print("Ứng viên chưa có trong từ điển:", [w for w, _ in cand[:150] if w not in td][:80])

Chưa có tiki_reviews.csv → bỏ qua phụ lục.
